<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent2.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Lab 1 - PART 1 - Wi-Fi Assistant</h2>
            <span style="color:#ff7800;">Wi-Fi AI assistant with tools<br/>
            This is the Wi-Fi AI Assistant lab.<br/>
            The ultimate goal of this deep dive.<br/>
            The main idea is to chat with an agent, then it does the promised tasks such as save Python codes, creating and working with MCP servers, etc.<br/>
            Since this lab will have many tools, it's sectionized into different parts.<br/>
            This part (Part 1) will include Wi-Fi assistant and Programmer agents.
            </span>
        </td>
    </tr>
</table>

## Wi-Fi Assistant

Let's make our MCP Server + Wi-Fi Assistant AI Agent

Here is a reference link to Model Context Protocol website which I used to create most of this lab:

https://modelcontextprotocol.io/docs/2026-07-28/getting-started/intro

#### Here is an overall view of this lab<br>
| Family | Key | Does | Does not |
| --- | --- | --- | --- |
| Ops | `wifi` | Talk Wi-Fi, list files in `Files/`, refuse off-topic | Write MCP servers, call Mist/NetBeez |
| Ops | `explorer` | Run `Codes/fill_explorer_checklist.py` on a scan CSV | Invent Python, call MCP |
| Ops | `ekahau` | Run known Ekahau scripts in `Codes/` | Author new MCP servers |
| Ops | `hamina` | Read Hamina / OpenIntent in `Files/` | Call live controllers |
| Author | `programmer` | Write/save Python (Mist server, explorer/ekahau/hamina helpers) | Live MCP calls |
| Local MCP | `mist` | Juniper Mist via **local** stdio MCP | Edit `Codes/` |
| Remote MCP | `netbeez` | NetBeez via **remote** MCP | Edit `Codes/` |

#### Benefits?
1. More deterministic!<br>
2. More cost-effective; you pick your LLM<br>
3. More secure; you can choose between cloud-based and local LLMs<br>
3. Only you allow one agent to talk/chat (the Wi-Fi).<br>
4. Learning how to cook instead of just feeding yourself! 


### This is how your orchestrator agent will look like at the end of this part; equipped with two agents.
#### Wi-Fi Assisstant & Programmer

<p align="center">
  <img src="../assets/routeragent1.png" width="600" />
</p>

The same as yesterday, let's import the necessary libraries:

In [81]:
from agents import Agent, WebSearchTool, trace, Runner, function_tool
from agents import OpenAIChatCompletionsModel, set_tracing_disabled
from agents.model_settings import ModelSettings
from agents.mcp import MCPServerStdio
from IPython.display import display
from dotenv import load_dotenv
from openai import AsyncOpenAI
from pydantic import BaseModel
from datetime import datetime
from typing import Literal
from pathlib import Path
import gradio as gr
import subprocess
import shlex
import json
import sys
import os

In [ ]:
load_dotenv(override=True)

In [83]:
# Constants 

MODEL_NAME1 = "gpt-5.4-mini"
MODEL_NAME2 = "gpt-5.4"
MODEL_NAME3 = "gpt-5.6-sol"

ollama = AsyncOpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
)
MODEL_NAME4 = OpenAIChatCompletionsModel(
    model="qwen2.5:7b",
    openai_client=ollama,
)

## Strategy for the AI Agent Assistant

Same idea as Lab 5: **separate agents**, then **orchestrate in Python** with **LLM** — not one agent that owns every tool. Which means tasks will be distributed between different agents.

1. **Wi-Fi Assistant** — chat about Wi-Fi, troubleshooting , web search about Wi-Fi questions 
2. **Programmer Agent** — write and save Python codes; save/run Python
3. **Explorer Checklist Agent** — read scan CSVs and fill `Checklist.csv`  
4. **Ekahau Agent** — runs Python files against esx files  
5. **hamina Agent** — read OpenIntent files and modify 
6. **Controller MCP Agent** — live WLAN API via the MCP server
7. **NetBeez MCP Agent** — live WLAN API via the MCP server (External MCP Server)  
8. **Orchestrator Agent** — no tools; it reads the chat and chooses one of the above agents 
 

Gradio calls an orchestrator that picks which agent to run.<br>
The Orchestrator agent reads the user’s query and based on the nature of each command it forwards them to the correct agent


## Agent1 : Wi-Fi Assistant Agent

In [84]:
FILES_DIR = Path("Files")

@function_tool
def list_files() -> str:
    """List every file in the Files/ folder, including csv, pcap, and pcapng captures."""
    folder = FILES_DIR.resolve()
    if not folder.is_dir():
        return f"Files folder not found at {folder}."
    files = sorted(p for p in folder.iterdir() if p.is_file() and not p.name.startswith("."))
    if not files:
        return f"No files in {folder}."
    return f"Files in {folder}:\n" + "\n".join(
        f"{p.name} ({p.stat().st_size} bytes)" for p in files
    )

TASKS = """
1. Chat normally about Wi-Fi, WLAN, LAN, and troubleshooting.
2. Use tools only when the user actually needs them:
- Web search: only Wi-Fi vendor/docs/topics/facts when they ask. Never for greetings. Never for off-topic questions.
- list_files: when they ask what is in the Files/ folder, or whether a csv, pcap, pcapng, or other local capture is there. Call list_files and answer only from that result. Never say you cannot see local files.
"""

INSTRUCTIONS = f"""
You are a friendly Wi-Fi assistant. Chat as a Wi-Fi helper.
You answer questions about Wi-Fi, WLAN, and troubleshooting.
You can see the local Files/ folder through list_files. Captures such as .csv, .pcap, and .pcapng live there.
When the user asks which files are present, call list_files before you answer.
Stay vendor-neutral. Do not name Juniper, Mist, Cisco, Aruba, or any other vendor unless the user names that vendor first.

Greetings and small talk: one short friendly reply.

Improtant Note: 
Off-topic (geography, astronomy, weather, currency, exchange rate, general trivia, news that is not Wi-Fi):
do not answer the fact. Do not use web search. Politely refuse and ask them to stay on Wireless LANs.

These are the tasks you can take on:
{TASKS}

"""

tools = [WebSearchTool(), list_files]

WiFi_assistant_agent = Agent(
    name="Wi-Fi Assistant Agent",
    instructions=INSTRUCTIONS,
    tools=tools,
    model=MODEL_NAME2,
    model_settings=ModelSettings(tool_choice="auto"),
)

## Agent2 : Programmer Agent

In [85]:
TASKS = """
1. Write Python when asked, and save it with save_python_code. Run it with run_saved_python when they ask you to execute it.
2. Build MCP servers in Python and save them with save_python_code so they can be used later.
"""

INSTRUCTIONS = f"""
You are an expert programmer.
You answer questions about Python, and MCP servers (how to build them).

These are the tasks you can take on:
{TASKS}

Use tools only when the user actually needs them:
- save_python_code: when they ask you to write a python code or an MCP server.
- run_saved_python: when they ask you to run a script in Codes/. Do not claim you cannot execute local Python.


If they ask you to build an MCP server, use the fastmcp framework and save it with save_python_code.
References:
https://modelcontextprotocol.io/llms-full.txt
https://github.com/modelcontextprotocol/python-sdk
"""

In [86]:
CODES_DIR = Path("Codes")

@function_tool
def save_python_code(filename: str, code: str) -> str:
    """Save Python code the assistant wrote into the Codes folder.

    filename: a short .py name, for example wifi_scan.py
    code: the full Python source to write
    """
    name = Path(filename).name
    if not name.endswith(".py"):
        name = name + ".py"
    path = CODES_DIR / name
    path.write_text(code, encoding="utf-8")
    print(f"Saved Python code to {path}")
    return f"Saved {path}"


@function_tool
def run_saved_python(filename: str, extra_args: str = "") -> str:
    """Run a .py file from the Codes folder and return stdout/stderr.

    filename: the file name only, for example fill_explorer_checklist.py
    extra_args: optional CLI args, for example Scan_20260915_111910.csv
    """

    name = Path(filename).name
    if not name.endswith(".py"):
        name = name + ".py"
    path = (CODES_DIR / name).resolve()
    if path.parent != CODES_DIR.resolve():
        return "Refusing to run a file outside Codes/"
    if not path.is_file():
        return f"No such file: {path}"
    extra = shlex.split(extra_args) if extra_args.strip() else []
    completed = subprocess.run(
        [sys.executable, str(path), *extra],
        capture_output=True,
        text=True,
        timeout=90,
        cwd=str(CODES_DIR.resolve().parent),
        env={**os.environ},
    )
    out = (completed.stdout or "") + (completed.stderr or "")
    return f"exit_code={completed.returncode}\n{out[-4000:]}"



tools = [WebSearchTool(), save_python_code, run_saved_python]
programmer_assistant_agent = Agent(
    name="Programmer Assistant Agent",
    instructions=INSTRUCTIONS,
    tools=tools,
    model=MODEL_NAME3,
    model_settings=ModelSettings(tool_choice="auto"),
)

## Chat with your agent

#### Part 1
Every Gradio message goes to the **Orchestrator**, which chooses `wifi` or `programmer`. Python then `Runner.run()`s that specialist with the **full chat history**.

`record_client_command` only logs prompts. Do **not** define `chat()` here — the Orchestrator cell below is the only `chat()` Gradio should use.




In [87]:
# Prompt log only. Gradio chat() is defined in the Orchestrator cell below.

def record_client_command(command):
    print(f"Recording client command: {command}")
    entry = {"time": datetime.now().isoformat(timespec="seconds"), "command": str(command)}
    with open("user-prompts.json", "a", encoding="utf-8") as f:
        f.write(json.dumps(entry) + "\n")


## Agent3 : Orchestrator Agent
### (An orchestrator between the other agents)

`pick_agent()` only scanned the last sentence for strings like `mist` then assigns the task to the proper agent.
Instead:

1. **Orchestrator agent** — LLM, no tools. Its instructions include each specialist’s own instructions. Structured output: `wifi` | `programmer`
2. It sees the **full Gradio chat**
3. Python then `Runner.run()` on that specialist, with the **same** chat so the specialist can follow the thread


In [ ]:
# Lab 5: LLM chooses, Python calls Runner.run() on that agent.

class RouteChoice(BaseModel):
    agent: Literal["wifi", "programmer"]
    reason: str


SPECIALISTS = {
    "wifi": WiFi_assistant_agent,
    "programmer": programmer_assistant_agent,
}

ORCHESTRATOR_INSTRUCTIONS = f"""
You are a router (Orchestrator). Read the conversation and choose the right specialist.

Match the conversation (all turns, not just the last line) to the specialist
whose instructions fit the work. Follow-ups stay with the same specialist

Important:
If the user asks question not related to Wi-Fi or programming, respond with a short answer and say that you are only able to help with your {SPECIALISTS.keys()}.
Questions about files in the Files/ folder (csv, pcap, pcapng, or "what files do I have") → wifi. That agent calls list_files.

wifi ({WiFi_assistant_agent.name}):
{WiFi_assistant_agent.instructions}

programmer ({programmer_assistant_agent.name}):
{programmer_assistant_agent.instructions}
"""

#Choosing a more powerful model to make the router more accurate

orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_INSTRUCTIONS,
    model=MODEL_NAME1,
    output_type=RouteChoice,
)


def _text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, dict):
        return _text(content.get("text") or content.get("content") or "")
    if isinstance(content, list):
        return "\n".join(part for part in (_text(x) for x in content) if part)
    return str(content).strip()


def conversation_input(message, history) -> list[dict]:
    items: list[dict] = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role") or "user"
            if role not in ("user", "assistant"):
                role = "user"
            text = _text(item.get("content") or item.get("text"))
            if text:
                items.append({"role": role, "content": text})
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            u, a = _text(item[0]), _text(item[1])
            if u:
                items.append({"role": "user", "content": u})
            if a:
                items.append({"role": "assistant", "content": a})
    text = _text(message) or str(message)
    items.append({"role": "user", "content": text})
    return items


async def run_wifi_assistant(items):
    result = await Runner.run(WiFi_assistant_agent, items)
    return result.final_output


async def run_programmer(items):
    result = await Runner.run(programmer_assistant_agent, items)
    return result.final_output


RUNNERS = {
    "wifi": run_wifi_assistant,
    "programmer": run_programmer,
}


async def chat(message, history):
    """Orchestrator LLM + specialist — full chat, not keyword pick_agent."""
    record_client_command(_text(message) or str(message))
    items = conversation_input(message, history)
    try:
        # Two traces on purpose: routing vs the specialist that answers.
        with trace("Orchestrator"):
            routed = await Runner.run(orchestrator_agent, items)
        choice = routed.final_output
        key = choice.agent if isinstance(choice, RouteChoice) else "wifi"
        reason = choice.reason if isinstance(choice, RouteChoice) else ""
        label = SPECIALISTS[key].name
        print(f"Orchestrator → {label} ({reason})")
        print(f"Conversation turns sent: {len(items)}")
        with trace(label):
            output = await RUNNERS[key](items)
        return output or "(The model returned no text. Open the notebook for the traceback.)"
    except Exception as e:
        print(f"chat() failed: {type(e).__name__}: {e}")
        return f"Error: {type(e).__name__}: {e}"


print("Orchestrator is an LLM. pick_agent keywords are gone.")
print("Specialists:", {k: a.name for k, a in SPECIALISTS.items()})


In [ ]:
# This launch uses chat() from the Orchestrator cell above.
if "Orchestrator LLM" not in (getattr(chat, "__doc__", "") or ""):
    raise RuntimeError("Run the Orchestrator cell above before launching Gradio.")

gr.ChatInterface(chat).launch(inbrowser=True)

## some sample prompts
#### Note: Double click on the text to be able to copy it. 


Write a Python code which can count from 0 to 10000000000 in two parallel processes.

What's the latest Extreme Network AP (Access Point) model?

List Juniper Mist 6GHz (Wi-Fi 6e and Wi-Fi7) capable Access Point? 

## The prompt to make an MCP Server
#### Note: Double click on the text to be able to copy it. 


Build me an MCP server for Juniper Mist API calls and save it with save_python_code as mist_mcp_server.py. Do not run it. Do not call Mist. Do not invent extra tools, helpers, async functions, or a second HTTP client. Reply with the saved path only.

The standalone fastmcp package is not installed. Import FastMCP from mcp.server.fastmcp. Do not write `from fastmcp import FastMCP`. Do not use async def or httpx.AsyncClient. Do not pass tags= to @mcp.tool. Those choices exit the process or deadlock stdio, and the notebook then reports Connection closed.

Write one module whose docstring says it exposes common Juniper Mist API operations, and that it must use mcp.server.fastmcp because the standalone fastmcp package crashes on import.

Imports: future annotations, os, re, Path, Any, Literal, httpx, load_dotenv, and FastMCP.

Startup, before any tool:
- load_dotenv on Path(__file__).resolve().parents[2] / ".env" with override=True. That path is the repo-root .env.
- mcp = FastMCP("Juniper Mist")
- Do not raise at import when the token is missing. Raise only inside the tool that needs it.
- Base URL, with any trailing slash removed: MIST_API_HOST, otherwise MIST_BASE_URL, otherwise MIST_API_BASE_URL, otherwise https://api.mist.com. Keep that value as MIST_BASE_URL.
- Read MIST_API_TOKEN, MIST_ORG_ID, and MIST_SITE_ID from the environment. Default each to an empty string, then strip it.

Helpers. All of them are ordinary functions.

- If MIST_API_TOKEN is empty, raise RuntimeError("MIST_API_TOKEN is not configured").
- If MIST_ORG_ID is empty, raise RuntimeError("MIST_ORG_ID is not configured").
- Normalize a caller path: strip it; an empty path raises ValueError("Mist API path cannot be empty"); a path starting with http://, https://, or // raises ValueError("Use a relative Mist API path, not an absolute URL"); add a leading slash when it is missing; if it does not start with /api/, prefix /api/v1.
- Error detail is the JSON body when the body is JSON, otherwise the raw text, cut at 2000 characters.
- One request helper, parameters method, path, params=None, body=None. Authorization is "Token <MIST_API_TOKEN>", not Bearer. Accept is application/json and application/vnd.api+json. Set Content-Type to application/json only when body is not None. Use one synchronous httpx.Client with base_url=MIST_BASE_URL, timeout of 30 seconds, follow_redirects=False, and trust_env=False. Send the uppercased method, the normalized path, params, and json=body. A network error raises RuntimeError("Mist API request failed: ..."). An HTTP error status raises RuntimeError("Mist API returned HTTP {status} for {METHOD} {path}: {detail}"). Status 204 or an empty body returns {"ok": True, "status_code": <status>}. Otherwise return parsed JSON. If the body is not JSON, return status_code, content_type, and text.
- Resolve a site id from the argument when it is non-empty, otherwise from MIST_SITE_ID. When both are empty, GET /orgs/{org_id}/sites with limit 1000 and page 1. A non-list response raises RuntimeError("Mist returned an unexpected response while resolving the site"). No sites raises RuntimeError("No sites were found in the configured Mist organization"). More than one site raises RuntimeError("Multiple sites exist; provide site_id or set MIST_SITE_ID. Sites: ...") and lists each site as "name (id)". A site with no id raises RuntimeError("The sole Mist site did not include an id").

Allowed stats types, and no others: clients, devices, mxedges, ports, sites, switches, unconnected_clients.

Tools. Decorate each with @mcp.tool(). Use the docstring below as that tool's one-line docstring. Do not add mist_create_wlan or any other tool.

- mist_raw_request(method, path, params=None, body=None). method is one of GET, POST, PUT, PATCH, DELETE. Docstring: Send an authenticated request to a relative Juniper Mist API path. Call the request helper.
- mist_get_org(). Docstring: Get details for the configured Juniper Mist organization. GET /orgs/{org_id}.
- mist_get_site(site_id=None). Docstring: Get details for a Juniper Mist site by ID or the configured default. GET /sites/{resolved site id}.
- mist_list_sites(). Docstring: List all sites in the configured Juniper Mist organization. Page GET /orgs/{org_id}/sites with limit 1000, starting at page 1. If a page is not a list, return that page. Stop when a page is shorter than 1000. Return the combined list.
- mist_list_wlans(site_id=None). Docstring: List WLANs for a Juniper Mist site by ID or the configured default. GET /sites/{resolved site id}/wlans.
- mist_get_site_stats(site_id=None, stats_type="devices"). Docstring: Get a supported statistics collection for a Juniper Mist site. Lowercase and strip stats_type. If it is outside the allowed set, or it is not only lowercase letters and underscores, raise ValueError("Unsupported stats_type '{stats_type}'; allowed values: {allowed}") with the allowed names sorted. GET /sites/{resolved site id}/stats/{type}.
- mist_update_wlan(site_id, wlan_id, update). Docstring: Update a WLAN at a Juniper Mist site using a partial configuration object. A blank site_id raises ValueError("site_id is required"). A blank wlan_id raises ValueError("wlan_id is required"). An empty update raises ValueError("update must contain at least one WLAN field"). PUT /sites/{site_id}/wlans/{wlan_id} with update as the JSON body.

End the file with mcp.run(transport="stdio") under if __name__ == "__main__". Do not call mcp.run() with no arguments.


## Did you notice?
### We didn't, don't, and won't say **Hi**, **Thank you** or the other similar things to our agnets?
## Are we mean? 😜
### Think of it before heading forward!

### As always, take a look at the trace

https://platform.openai.com/traces

### Note: Do not close the chat, let's go to the next part.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent2.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Lab 1 - PART 2 - MCP Testing and adding more!</h2>
            <span style="color:#ff7800;">Two new agents: Explorer checklist, and live controller MCP.<br/>
            Chat application (Gradio application) reads user messages, and routes them to the proper agent through the Orchestrator agent.<br/>
            This orchestration is happening by AI. And, in this section (Part 2) two new agents will be added.
            </span>
        </td>
    </tr>
</table>


### This is how your orchestrator agent will look like at the end of this part; equipped with two new agents.
#### Wi-Fi Explorer Checklist & Mist MCP Server

<p align="center">
  <img src="../assets/routeragent2.png" width="600" />
</p>

## Objectives

### 1. Building and Connecting the Wi-Fi Explorer to the Orchestrator agent


The main purpose of this section is use the Wi-Fi explorer csv output to fill in a checklist.

We use the Programmer agent to write the checklist code.

### 2. Connect the MCP server to the Orchestrator agent

Saving a `.py` file is not enough. The **Controller MCP Agent** only sees MCP tools if we **start the server** and pass it as `mcp_servers`.

The OpenAI Agents SDK does that with `MCPServerStdio`. It launches `Codes/mist_mcp_server.py` (the file you generated in class) and lists its tools.



## Prompt — paste to Gradio (Programmer)
#### Note: Double click on the text to be able to copy it. 




Write a standalone Python script and save it with save_python_code as fill_explorer_checklist.py. Do not run it. Do not call MCP. Do not invent extra features.

This script is the only implementation of the Explorer checklist. Another agent will execute it later. Behavior must match the lab’s current fill_explorer_checklist tool exactly.

CLI
- Usage: python fill_explorer_checklist.py <scan_filename> <network_name>
- scan_filename is a .csv name in Explorer/ (basename only; ignore any path).
- network_name is the SSID; match case-insensitive substring on column "Network Name".
- If argv is wrong, print an error and exit 1.
- Working directory when run will be the lab folder (parent of Codes/), so use Path("Explorer").

Input / output files
- Read Explorer/<scan_filename>. If missing, print: No such scan CSV in Explorer/: <name>
- Never overwrite Explorer/Checklist.csv.
- Write the next unused file: Explorer/Checklist_filled(1).csv, then (2), (3), …
- UTF-8. Scan files may have a BOM (utf-8-sig). Skip leading comment lines that are empty, start with #, or start with WiFi_Status: before DictReader.

Scan CSV columns (Wi-Fi Explorer Pro 3)
Network Name, Band, Channel, Channel Width, Signal (dBm), BSSID, AP Name, Vendor, Security, Mode, Max Rate (Mbps), Amendments, Country Code, Transmit Power, Channel Utilization (%)

Fill rules
1. Filter rows where Network Name contains network_name (case-insensitive). If none: No Network Name matching '<ssid>' in <file>.
2. Group remaining rows by band key from Band:
   - "2.4" if Band contains 2.4 or 2,4
   - else "6" if Band contains 6
   - else "5" if Band contains 5
   Skip rows with no band key.
3. Keep one row per band: strongest Signal (dBm) (larger number is stronger, e.g. -50 beats -80). Parse the first integer/float in the cell.
4. Output bands in order 2.4, then 5, then 6. Omit missing bands.
5. Copy fields into checklist columns. Parse Amendments by splitting on / or comma. Columns k, v, r get the letter if that amendment is present, else empty.
6. Append marks to the value cells (space + mark), do not replace the original number/text:
   Signal: 🟢 if rssi >= -65; ⚠️ if -73 <= rssi <= -70; 🔴 if rssi <= -74; ⚠️ otherwise (including -66 to -69).
   2.4 GHz Channel: 🟢 if 1, 6, or 11 else 🔴.
   5 GHz Channel: append (UNII-1) for {36,40,44,48}; (UNII-2) for {52,56,60,64,100,104,108,112,116,120,124,128,132,136,140,144}; (UNII-3) for {149,153,157,161,165}.
   6 GHz Channel: PSC 🟢 if channel in {5,21,37,53,69,85,101,117,133,149,165,181,197,213,229} else non-PSC ⚠️.
   Channel Width (first int in the cell): 2.4 → 🟢 if 20 else 🔴; 5 → 🟢 if 20 or 40, ⚠️ if 80, else 🔴; 6 → 🟢 if 80 else ⚠️.

Output CSV
- First write these comment lines exactly:

# Wi-Fi Explorer checklist
# Status:  🟢 Good    ⚠️ Warning - Consider a Review   🔴 Bad 
# Signal:  🟢 ≥ -65 dBm    ⚠️ -70 to -73    🔴 ≤ -74
# 2.4 GHz channel:  🟢 1 / 6 / 11    🔴 other
# Width:  2.4 🟢 20    5 🟢 20/40  ⚠️ 80    6 🟢 80 ⚠️ 160 and 🔴 other
# 5 GHz: UNII-1 / UNII-2 / UNII-3     6 GHz: 🟢 PSC    ⚠️ non-PSC

- Then header and rows with columns exactly:
BSSID, AP Name, Network Name, Vendor, Channel, Signal (dBm), Band, Channel Width, Security, Mode, Max Rate (Mbps), k, v, r, Country Code, Transmit Power, Channel Utilization (%)

Stdout on success
Wrote Checklist_filled(N).csv (M row(s) for '<ssid>' from <scan>).
Then one preview line per output row: col=value joined with " | " using the checklist columns.

Constraints
- stdlib only (csv, re, pathlib, sys). No pandas.
- Do not guess missing scan values.
- Save the full file with save_python_code. Reply with the saved path only.

## Agent4 : Explorer Checklist Agent
### Wi-Fi Explorer Pro 3 files

Here we build our agent which works as mentioned above.


In [ ]:
# EXPLORER_DIR = Path("Explorer")


# @function_tool
# def list_explorer_csvs() -> str:
#     """List CSV files in the Explorer/ folder."""
#     files = sorted(EXPLORER_DIR.glob("*.csv"))
#     if not files:
#         return "No CSV files in Explorer/. Export from Wi-Fi Explorer Pro 3 into that folder."
#     return "CSV files in Explorer/:\n" + "\n".join(f"{p.name} ({p.stat().st_size} bytes)" for p in files)


# EXPLORER_INSTRUCTIONS = """
# You fill Wi-Fi Explorer Pro 3 scan CSVs into the checklist by running an existing script.
# You do not write, edit, or save Python. The Programmer already saved Codes/fill_explorer_checklist.py.

# SSID is column Network Name. Never overwrite Checklist.csv.
# If the scan filename is missing, call list_explorer_csvs.
# Then call run_saved_python with:
# - filename: fill_explorer_checklist.py
# - extra_args: <scan_filename> <network_name>
# If the Network Name has spaces, quote it: Scan_20260915_111910.csv "My SSID"

# Each run saves a new file: Explorer/Checklist_filled(1).csv, then (2), (3), and so on.
# If the script is missing, tell the user to ask the Programmer to write fill_explorer_checklist.py first.
# Do not guess scan values. Do not call live controller APIs.
# """

# explorer_agent = Agent(
#     name="Explorer Checklist Agent",
#     instructions=EXPLORER_INSTRUCTIONS,
#     tools=[list_explorer_csvs, run_saved_python],
#     model=MODEL_NAME1,
#     model_settings=ModelSettings(tool_choice="auto"),
# )
# print("Explorer agent tools:", [t.name for t in explorer_agent.tools])
# print("Folder:", EXPLORER_DIR.resolve())



## Agent4 : Explorer Checklist Agent - with Local (Open Source) LLM
### Wi-Fi Explorer Pro 3 files

#### Ollama
1. Download and install Ollama from ollama.com, then open the app.<br>
2. In Terminal, pull the model and confirm it is there:<br>

ollama pull qwen2.5:7b;

ollama list;

curl http://127.0.0.1:11434


In this way: The Explorer agent’s model stays on your machine. Its prompt and tool results go to http://127.0.0.1:11434, and run_saved_python reads the CSV on disk. Ollama does not send that chat to OpenAI.

The rest of this notebook still does. Gradio sends every message to the orchestrator first, and that orchestrator is still an OpenAI model (MODEL_NAME1, MODEL_NAME2, or MODEL_NAME3, depending on which cell you launched). OpenAI sees the text you type, including a filename and an SSID, so it can choose the Explorer agent. The scan rows themselves are read later, inside the local tool call.

set_tracing_disabled(True) stops this kernel from uploading traces to OpenAI. A route to the Mist or NetBeez agent still calls those cloud APIs.

### Benefits?

An on-machine open-source model keeps that agent’s work on the computer that is already holding the files.

**The chat stays local.** For the Explorer agent, the prompt, the CSV names, and the tool results go to 127.0.0.1:11434. They are not sent to OpenAI for that model call.<br>

**There is no token bill and no API key** for that agent. Once qwen2.5:7b is pulled, running it does not depend on an OpenAI account or a rate limit.<br>

**It still runs when the network is down.** A lab, a classroom, or a site survey can use the agent without an internet connection.<br>

**You can change models yourself.** ollama pull another model and point OpenAIChatCompletionsModel at the new name. The agent code stays the same.<br>

**The same setup is easier to share.** Someone else can run the notebook with Ollama and no OpenAI key for that agent.<br>


In [ ]:
# Replacing OpenAI model with Ollama
# qwen2.5:7b was a practical default for the Explorer agent, not a model measured on your scans.
# llama3.1 is a valid alternative. The name in OpenAIChatCompletionsModel has to match whatever ollama list shows after you pull it.

set_tracing_disabled(True)  # traces are sent to OpenAI
EXPLORER_DIR = Path("Explorer")


@function_tool
def list_explorer_csvs() -> str:
    """List CSV files in the Explorer/ folder."""
    files = sorted(EXPLORER_DIR.glob("*.csv"))
    if not files:
        return "No CSV files in Explorer/. Export from Wi-Fi Explorer Pro 3 into that folder."
    return "CSV files in Explorer/:\n" + "\n".join(f"{p.name} ({p.stat().st_size} bytes)" for p in files)


EXPLORER_INSTRUCTIONS = """
You fill Wi-Fi Explorer Pro 3 scan CSVs into the checklist by running an existing script.
You do not write, edit, or save Python. The Programmer already saved Codes/fill_explorer_checklist.py.

SSID is column Network Name. Never overwrite Checklist.csv.
If the scan filename is missing, call list_explorer_csvs.
Then call run_saved_python with:
- filename: fill_explorer_checklist.py
- extra_args: <scan_filename> <network_name>
If the Network Name has spaces, quote it: Scan_20260915_111910.csv "My SSID"

Each run saves a new file: Explorer/Checklist_filled(1).csv, then (2), (3), and so on.
If the script is missing, tell the user to ask the Programmer to write fill_explorer_checklist.py first.
Do not guess scan values. Do not call live controller APIs.
"""

explorer_agent = Agent(
    name="Explorer Checklist Agent",
    instructions=EXPLORER_INSTRUCTIONS,
    tools=[list_explorer_csvs, run_saved_python],
    model=MODEL_NAME4,
    model_settings=ModelSettings(tool_choice="auto"),
)
print("Explorer agent tools:", [t.name for t in explorer_agent.tools])
print("Folder:", EXPLORER_DIR.resolve())

## Agent5 : MCP Agent

In [ ]:
if "mcp_server" in globals():
    try:
        await mcp_server.cleanup()
    except Exception:
        pass

server_path = str(Path("Codes/mist_mcp_server.py").resolve())
mcp_params = {
    "command": sys.executable,
    "args": [server_path],
    "env": {**os.environ},
}

mcp_server = MCPServerStdio(
    params=mcp_params,
    client_session_timeout_seconds=120,
    cache_tools_list=True,
)
await mcp_server.connect()

mcp_tools = await mcp_server.list_tools()
print("MCP tools from Codes/mist_mcp_server.py:")
for tool in mcp_tools:
    print(f"  - {tool.name}: {tool.description}")

CONTROLLER_INSTRUCTIONS = """
You call the connected MCP server for live WLAN controller data and config.
Tools: mist_list_sites, mist_get_org, mist_get_site, mist_list_wlans, mist_get_site_stats,
mist_create_wlan, mist_update_wlan, mist_raw_request (to create or delete WLANs).

You can CREATE a new SSID, update, or delete an existing SSID.

Important:
- Do not re-write on an existing SSID once you're asked to create a new one.
- Create with mist_raw_request POST /api/v1/sites/{site_id}/wlans.
- Delete with mist_raw_request DELETE /api/v1/sites/{site_id}/wlans/{wlan_id}.
- Update an existing SSID with mist_update_wlan. Do not create a second SSID to change bands or security.

Mist ignores any other names for these fields:
- All three radios: bands ["24", "5", "6"]. 2.4 GHz is "24", never "2.4".
- WPA3-Personal (SAE): auth.type "psk", auth.psk the passphrase, auth.pairwise ["wpa3"].
  Do not send pairwise "ccmp", and do not send auth.suite or auth.akm.

After every create or update, call mist_list_wlans and read that WLAN's bands and auth.pairwise.
It is done only when bands is ["24", "5", "6"] and auth.pairwise is ["wpa3"]. Report those two values.
"""

mcp_agent = Agent(
    name="Controller MCP Agent",
    instructions=CONTROLLER_INSTRUCTIONS,
    model=MODEL_NAME2,
    model_settings=ModelSettings(tool_choice="auto"),
    mcp_servers=[mcp_server],
)
print("Local MCP server is connected. 🍻")


### Where to put the Mist API Token?
#### Note: This step is already done for this lab using the provided tokens.
#### The instructions below are for later use, if you want to replace the lab token with your own API key.

Put it in the same .env file as OPENAI_API_KEY (repo root, gitignored). Do not put it in the notebook, in chat, or in Codes/.

MIST_API_TOKEN=paste-your-token-here

load_dotenv() in the notebook is not enough for MCP. The API server is a separate process. Either:

* pass the env when you start MCPServerStdio, or
* call load_dotenv() inside mcp_api_server.py


## Agent3 : Orchestrator Agent
### (Orchestrator Updates)

`pick_agent()` only scanned the last sentence for strings like `mist`. That cannot follow “SSIDs and timezone” after a sites question.

Instead:

1. **Orchestrator agent** — LLM, no tools. Its instructions include each specialist’s own instructions. Structured output: `wifi` | `programmer` | `explorer` |`controller (MCP)`
2. It sees the **full Gradio chat**
3. Python then `Runner.run()` on that specialist, with the **same** chat so the specialist can follow the thread


In [ ]:
# Lab 5: LLM chooses, Python calls Runner.run() on that agent.

class RouteChoice(BaseModel):
    agent: Literal["wifi", "programmer", "explorer", "controller"]
    reason: str


SPECIALISTS = {
    "wifi": WiFi_assistant_agent,
    "programmer": programmer_assistant_agent,
    "explorer": explorer_agent,
    "controller": mcp_agent,
}

ORCHESTRATOR_INSTRUCTIONS = f"""
You are a router (Orchestrator). Read the conversation and choose the right specialist.
Choose exactly one specialist for this turn.

Match the conversation (all turns, not just the last line) to the specialist
whose instructions fit the work. Follow-ups stay with the same specialist
when they continue that job ("the list", "SSIDs", "do an API call", "from the MCP server").


Important:
if the user asks question not related to Wi-Fi or programming, respond with a short answer and say that you are only able to help with your {SPECIALISTS.keys()}.
Listing files in Files/ (csv, pcap, pcapng, or "what files do I have") → wifi. That agent calls list_files.
Explorer only fills a checklist from a named scan CSV in Explorer/. Do not send a general file search to explorer.


wifi ({WiFi_assistant_agent.name}):
{WiFi_assistant_agent.instructions}

programmer ({programmer_assistant_agent.name}):
{programmer_assistant_agent.instructions}

explorer ({explorer_agent.name}):
{explorer_agent.instructions}

controller ({mcp_agent.name}):
{mcp_agent.instructions}
"""

#Choosing a more powerful model to make the router more accurate

orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_INSTRUCTIONS,
    model=MODEL_NAME2,
    output_type=RouteChoice,
)


def _text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, dict):
        return _text(content.get("text") or content.get("content") or "")
    if isinstance(content, list):
        return "\n".join(part for part in (_text(x) for x in content) if part)
    return str(content).strip()


def conversation_input(message, history) -> list[dict]:
    items: list[dict] = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role") or "user"
            if role not in ("user", "assistant"):
                role = "user"
            text = _text(item.get("content") or item.get("text"))
            if text:
                items.append({"role": role, "content": text})
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            u, a = _text(item[0]), _text(item[1])
            if u:
                items.append({"role": "user", "content": u})
            if a:
                items.append({"role": "assistant", "content": a})
    text = _text(message) or str(message)
    items.append({"role": "user", "content": text})
    return items


async def run_wifi_assistant(items):
    result = await Runner.run(WiFi_assistant_agent, items)
    return result.final_output

async def run_programmer(items):
    result = await Runner.run(programmer_assistant_agent, items)
    return result.final_output

async def run_explorer(items):
    result = await Runner.run(explorer_agent, items)
    return result.final_output

async def run_controller(items):
    result = await Runner.run(mcp_agent, items)
    return result.final_output


RUNNERS = {
    "wifi": run_wifi_assistant,
    "programmer": run_programmer,
    "explorer": run_explorer,
    "controller": run_controller,
}


async def chat(message, history):
    """Orchestrator LLM + specialist — full chat, not keyword pick_agent."""
    record_client_command(_text(message) or str(message))
    items = conversation_input(message, history)
    try:
        with trace("Orchestrator"):
            routed = await Runner.run(orchestrator_agent, items)
        choice = routed.final_output
        key = choice.agent if isinstance(choice, RouteChoice) else "wifi"
        reason = choice.reason if isinstance(choice, RouteChoice) else ""
        label = SPECIALISTS[key].name
        print(f"Orchestrator → {label} ({reason})")
        print(f"Conversation turns sent: {len(items)}")
        with trace(label):
            output = await RUNNERS[key](items)
        return output or "(The model returned no text. Open the notebook for the traceback.)"
    except Exception as e:
        print(f"chat() failed: {type(e).__name__}: {e}")
        return f"Error: {type(e).__name__}: {e}"


print("Orchestrator is an LLM. pick_agent keywords are gone.")
print("Specialists:", {k: a.name for k, a in SPECIALISTS.items()})


#### You can close the old chat now!


In [ ]:
# Interrupt any old Gradio first. This launch uses chat() from the Orchestrator cell above.
if "Orchestrator LLM" not in (getattr(chat, "__doc__", "") or ""):
    raise RuntimeError("Run the Orchestrator cell above before launching Gradio.")

# Close the previous chat so the old orchestrator cannot keep answering on another port.
gr.close_all()
gr.ChatInterface(chat).launch(inbrowser=True)


### Some Example Requests:
#### Note: Double click on the question to be able to copy them.



* Can you check my Juniper Mist Dashboard through my MCP server and list my sites?
* Provide more information the "OREA Hotel" site.
* Check the WLANs (SSIDs) in "OREA Hotel" site and list them for me with details.
* Are there any active clients in this site right now?
* Provide my clients list?
* Create a new WLAN under the same site as the following:
1. The SSID “WLPC_DEEPDIVE” 
2. It broadcasts on all three frequency (radio) bands
3. Security type is WPA3/SAE and the required passphrase is: "WLPCPRG2026"
* Disable 2.4GHz on “WLPC_DEEPDIVE” SSID (2.4GHz might come back as 24 in API calls)


#### Steps before going forward

1. Export from Wi-Fi Explorer Pro 3 → Explorer/ as CSV.<br>

<td style="width: 556px; height: 336px; vertical-align: middle;">
    <img src="../assets/explorer.png" width="500" height="300" style="display: block;" />
</td>

2. Ask the Programmer to write Codes/fill_explorer_checklist.py (prompt below). Confirm it saved the file; do not ask Programmer to fill the checklist.<br>
3. Ask the Explorer to run that script for a named scan + Network Name. Explorer must not write Python.<br>
4. Output is Explorer/Checklist_filled(1).csv, then (2), (3), … Never overwrite Checklist.csv.<br>

#### Run a scan with Wi-Fi Explorer Pro3. Then, click on Files, Export all Networks, and store it as a csv file in the Explorer folder within these project files. Then, follow with commands.

* What scan CSV files are in the Explorer folder?
* Use that scan file (file name here) and fill the checklist for SSID "WLPC_DEEPDIVE". Run fill_explorer_checklist.py
* Delete the WLAN “WLPC_DEEPDIVE”

### Don't close the chat, let's go to the next part.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent2.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Lab 1 - PART 3 - ESX and OpenIntent files</h2>
            <span style="color:#ff7800;">Adding more agents to help us with more wireless enginrers daily tasks.<br/>
            1. Sometimes, the scripts to work with a software already exist. Hence, you just need an agent which reads your command, use the script and use them. We use this idea for the Ekahau agent.<br/>
            2. Sometimes, there's a way to work with a file with some scripts. A programmer agent can write the codes, and then your agent can read your request and run them once you ask. 
            </span>
        </td>
    </tr>
</table>


### This is how your orchestrator agent will look like at the end of this part; equipped with two new agents.
#### Ekahau & Hamina OpenIntent

<p align="center">
  <img src="../assets/routeragent3.png" width="600" />
</p>

## Ekahau & Hamina

### Part 3

> A good amount of **Ekahau Python** files are already existing. First approach is to feed them to the existing Agent.<br>
> The second approach is to prompt and explain the OpenIntent files for **Hamina**

Note: By the time of creating this lab neither Hamina nor Ekahau has their own APIs.</br>
We need to build the Wi-Fi Assistant again by adding some more INSTRUCTION.</br>

### Note: Please place your .esx and OpenIntent file in the Files/ folder before starting this lab.

## Agent 6: Ekahau Agent

In [ ]:
FILES_DIR = Path("Files")


@function_tool
def list_esx_files() -> str:
    """List Ekahau .esx files in the Files/ folder."""
    files = sorted(FILES_DIR.glob("*.esx"))
    if not files:
        return "No .esx files in Files/. Put an Ekahau project there first."
    return "ESX files in Files/:\n" + "\n".join(
        f"{p.name} ({p.stat().st_size} bytes)" for p in files
    )


TASKS = """
1. Extract AP note images from an Ekahau .esx by running extract-AP-images.py.
2. Put .esx files in Files/. Output goes to AP-Images/ (one folder per floor).
"""

INSTRUCTIONS = f"""
You are an Ekahau assistant. You only run existing scripts in Codes/.
You do not write, edit, or save Python. New Ekahau scripts belong to the Programmer.

These are the tasks you can take on:
{TASKS}

Use tools only when the user actually needs them:
- If they did not name an .esx, or you are not sure which file they mean, call list_esx_files first. Use only a name that tool returns. Never guess a filename (including Office_Survey.esx).
- If there are no .esx files, tell them to put one in Files/. If there are several, ask which one.
- Then run_saved_python with filename extract-AP-images.py
- extra_args: Files/<exact name from list_esx_files>
- If the script or .esx is missing, say so. Do not invent a path.

Do not call live controller APIs. Do not touch Explorer CSVs or Hamina OpenIntent zips.
"""

ekahau_agent = Agent(
    name="Ekahau Agent",
    instructions=INSTRUCTIONS,
    tools=[list_esx_files, run_saved_python],
    model=MODEL_NAME1,
    model_settings=ModelSettings(tool_choice="auto"),
)
print("Ekahau agent tools:", [t.name for t in ekahau_agent.tools])
print("Folder:", FILES_DIR.resolve())


## Agent 7: OpenIntent (Hamina) Agent

Put a Hamina **zip** in `Files/`. The JSON inside follows **oi-wifi 2.0.0**:

https://github.com/google/openintent/blob/main/release/2.0.0/models/oi-wifi.schema.json

**AP vendor** in this lab is `accesspoints[].manufacturer` (oi-wifi schema: “The manufacturer name for this access point”).

Changes save as `{name}_modified.zip`.

#### Steps
1. Place a Hamina OpenIntent zip in Files/.<br>
2. Ask the Programmer to write Codes/update_openintent.py (prompt below). Confirm it saved the file; do not ask Programmer to edit the zip.<br>
3. Ask the OpenIntent agent to read or change APs. That agent must not write Python.<br>
4. Output is Files/{name}_modified.zip. Never overwrite the original zip.<br>


### Prompt — paste to Gradio (Programmer)
#### Note: Double click on the texts to be able to copy and paste.



Write a standalone Python script and save it with save_python_code as update_openintent.py. Do not run it. Do not call MCP. Do not invent extra features.

This script is the only implementation of Hamina OpenIntent zip read/update. Another agent will execute it later. Use oi-wifi 2.0.0 field names (https://github.com/google/openintent/blob/main/release/2.0.0/models/oi-wifi.schema.json).

CLI
- Working directory when run will be the lab folder (parent of Codes/), so use Path("Files").
- Usage:
  python update_openintent.py read <zip_name>
  python update_openintent.py update <zip_name> <ap_name> [options]
- zip_name is a .zip name in Files/ (basename only; ignore any path).
- If argv is wrong, print an error and exit 1.

Find the JSON inside the zip
- Open Files/<zip_name>. If missing or not a .zip, print: <name> not in Files/. Zips: <comma-separated names or (none)>
- Inner JSON: first zip member whose basename lowercases to start with "openintent" AND does not contain "warning".
- Parse that member as JSON (oi-wifi 2.0.0). AP vendor is accesspoints[].manufacturer.

oi-wifi 2.0.0 field names (use these exactly; do not invent keys)
- accesspoints[].name
- accesspoints[].manufacturer
- accesspoints[].model
- accesspoints[].floorplan_name
- Radios are ONLY accesspoints[].dot11_radios  (a list). NEVER use accesspoints[].radios — that key does not exist in Hamina OpenIntent and channel/tx changes will silently do nothing.
- Each radio dict looks like:
  {
    "id": 0,
    "radio_function": "CLIENT_ACCESS",
    "band": "FREQ_2.4GHZ",
    "channel": 1,
    "channel_width": "20_MHz",
    "transmit_power": 6,
    "mimo_chains": 2,
    "channel_assignment": "AUTOMATIC"
  }
  band values: FREQ_2.4GHZ, FREQ_5GHZ, FREQ_6GHZ


read
- Print: <zip name> → <inner json path>
- Then "AP facts:" and one line per AP:
  - <name>: vendor=<manufacturer or (no manufacturer)> model=<model> floor=<floorplan_name>
- Then a blank line, then "accesspoints:" and json.dumps(accesspoints, indent=2) (empty list if missing).

update
Options (all optional strings; only apply non-empty values):
  --model
  --floorplan_name
  --channel          (ALL radios; only if NO per-band flags are set)
  --transmit_power   (ALL radios; only if NO per-band flags are set)
  --channel_24 --tx_24 --channel_5 --tx_5 --channel_6 --tx_6
  --assignment       (ALL radios; only if NO per-band assignment flags are set)
  --assignment_24 --assignment_5 --assignment_6
  Each assignment value must be exactly AUTOMATIC or MANUAL. Anything else: print an error and exit 1.

Rules
- If none of model, floorplan_name, channel, transmit_power, assignment, or any per-band channel/tx/assignment flag is set, print:
  Pass model, floorplan_name, and/or per-band channel_24/tx_24, channel_5/tx_5, channel_6/tx_6, assignment_24/assignment_5/assignment_6.
- Match exactly one AP. Matching order:
  1. exact case-insensitive name
  2. else unique substring match
  3. else unique match on digits extracted from the query appearing in the AP name
  If none: No AP matching '<ap_name>'. APs: <names>
  If several: Several APs match '<ap_name>': <names>. Use the full name.
- Change ONLY that AP.
- Radios to edit: that AP's dot11_radios list only. Write channel_assignment on those objects. Never invent a radios key.
- A per-band flag of any kind (channel, tx, or assignment) means: do not apply --channel, --transmit_power, or --assignment to every radio.
- Per-band channel/tx: apply only to that band. Leave other bands' channel and transmit_power unchanged.
- Per-band assignment: set channel_assignment only on that band. Leave other bands' channel_assignment unchanged.
- No per-band flags: --channel and --transmit_power apply to every object in dot11_radios, and --assignment does the same for channel_assignment.
- Assignment value: uppercase AUTOMATIC or MANUAL only. Reject anything else.
- Changing assignment alone is valid. It does not require a channel or transmit_power change.
- Band from radio.band (uppercase): "24" if contains 2.4 or 2400; "6" if 6GHZ or FREQ_6; "5" if 5GHZ or FREQ_5.
- Set channel with int(); transmit_power with int(float()).
- Write Files/<stem>_modified.zip. Copy every zip member; replace only the OpenIntent JSON. Leave the original zip unchanged.
- On success print: Wrote <out.name> (left <original.name> unchanged). Then semicolon-separated notes like model=...; 24 ch=...; 24 tx=...; 24 assignment=MANUAL; 5 assignment=MANUAL

Constraints
- stdlib only (json, zipfile, pathlib, sys, argparse).
- Do not guess missing AP fields.
- Save the full file with save_python_code. Reply with the saved path only.

In [ ]:
FILES_DIR = Path("Files")

@function_tool
def list_openintent_files() -> str:
    """List Hamina / OpenIntent zip files in the Files/ folder."""
    files = sorted(FILES_DIR.glob("*.zip"))
    if not files:
        return "No zip files in Files/. Put a Hamina OpenIntent zip there first."
    return "Zip files in Files/:\n" + "\n".join(
        f"{p.name} ({p.stat().st_size} bytes)" for p in files
    )


INSTRUCTIONS = """
You read and change Hamina OpenIntent zips in Files/ by running an existing script.
You do not write, edit, or save Python. The Programmer already saved Codes/update_openintent.py.


JSON follows oi-wifi 2.0.0:
https://github.com/google/openintent/blob/main/release/2.0.0/models/oi-wifi.schema.json


If the zip filename is missing, call list_openintent_files. Use only a name that tool returns.
Then call run_saved_python with:
- filename: update_openintent.py
- extra_args: read <zip_name>
  or: update <zip_name> <ap_name> --channel_24 ... --tx_24 ... --assignment_24 MANUAL --channel_5 ... --tx_5 ... --assignment_5 MANUAL --channel_6 ... --tx_6 ... --assignment_6 MANUAL
- channel_assignment is AUTOMATIC or MANUAL on dot11_radios[]. Pass --assignment_24 / --assignment_5 / --assignment_6 only for bands the user named. Pass --assignment only when they want every radio changed and gave no per-band values.
- Changing assignment alone is valid. Do not also change channel or transmit_power unless they asked.

Each change writes Files/{name}_modified.zip. Never overwrite the original zip.
Do not call live controller APIs. Do not touch Explorer CSVs or Ekahau .esx files.
"""

Hamina_agent = Agent(
    name="Hamina Agent",
    instructions=INSTRUCTIONS,
    tools=[list_openintent_files, run_saved_python],
    model=MODEL_NAME2,
    model_settings=ModelSettings(tool_choice="auto"),
)

print("Hamina agent tools:", [t.name for t in Hamina_agent.tools])
print("Folder:", FILES_DIR.resolve())

#### Re-running the Orchestrator Agent again

1. **Orchestrator agent** — LLM, no tools. Its instructions include each specialist’s own instructions. Structured output: `wifi` | `programmer` | `explorer` |`controller`| `ekahau` | `Hamina` |
2. It sees the **full Gradio chat**
3. Python then `Runner.run()` on that specialist, with the **same** chat so the specialist can follow the thread

In [79]:
# Libraries are already imported at the top of the notebook


class RouteChoice(BaseModel):
    agent: Literal["wifi", "explorer", "controller", "programmer", "ekahau", "Hamina"]
    reason: str


SPECIALISTS = {
    "wifi": WiFi_assistant_agent,
    "programmer": programmer_assistant_agent,
    "explorer": explorer_agent,
    "ekahau": ekahau_agent,
    "controller": mcp_agent,
    "Hamina": Hamina_agent,
}

ORCHESTRATOR_INSTRUCTIONS = f"""
You are a router. You do not answer the user and you do not call APIs.
Choose exactly one specialist for this turn.

Match the conversation (all turns, not just the last line) to the specialist
whose instructions fit the work. Follow-ups stay with the same specialist
when they continue that job ("the list", "SSIDs", "do an API call", "from the MCP server",
"the zip", "OpenIntent", "AP model").
esx files are in Files/ folder are Ekahau files.
zip files are in Files/ folder are Hamina files.
Other files in Files/ (pcap, pcapng, csv captures, or "what is in Files") → wifi, which calls list_files.


Important:
If the user asks question not related to Wi-Fi or programming, respond with a short answer and say that you are only able to help with your {SPECIALISTS.keys()}.


wifi ({WiFi_assistant_agent.name}):
{WiFi_assistant_agent.instructions}

programmer ({programmer_assistant_agent.name}):
{programmer_assistant_agent.instructions}

explorer ({explorer_agent.name}):
{explorer_agent.instructions}

ekahau ({ekahau_agent.name}):
{ekahau_agent.instructions}

controller ({mcp_agent.name}):
{mcp_agent.instructions}

Hamina ({Hamina_agent.name}):
{Hamina_agent.instructions}

Ekahau .esx / extract AP images → ekahau. Hamina OpenIntent zip / AP model-channel-power in Files/ → openintent.
"""

#Choosing a more powerful model to make the router more accurate

router_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_INSTRUCTIONS,
    model=MODEL_NAME2,
    output_type=RouteChoice,
)


def _text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, dict):
        return _text(content.get("text") or content.get("content") or "")
    if isinstance(content, list):
        return "\n".join(part for part in (_text(x) for x in content) if part)
    return str(content).strip()


def conversation_input(message, history) -> list[dict]:
    items: list[dict] = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role") or "user"
            if role not in ("user", "assistant"):
                role = "user"
            text = _text(item.get("content") or item.get("text"))
            if text:
                items.append({"role": role, "content": text})
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            u, a = _text(item[0]), _text(item[1])
            if u:
                items.append({"role": "user", "content": u})
            if a:
                items.append({"role": "assistant", "content": a})
    text = _text(message) or str(message)
    items.append({"role": "user", "content": text})
    return items


async def run_wifi_assistant(items):
    result = await Runner.run(WiFi_assistant_agent, items)
    return result.final_output

async def run_programmer(items):
    result = await Runner.run(programmer_assistant_agent, items)
    return result.final_output

async def run_explorer(items):
    result = await Runner.run(explorer_agent, items)
    return result.final_output

async def run_controller(items):
    result = await Runner.run(mcp_agent, items)
    return result.final_output

async def run_Hamina(items):
    result = await Runner.run(Hamina_agent, items)
    return result.final_output

async def run_ekahau(items):
    result = await Runner.run(ekahau_agent, items)
    return result.final_output


RUNNERS = {
    "wifi": run_wifi_assistant,
    "explorer": run_explorer,
    "controller": run_controller,
    "Hamina": run_Hamina,
    "ekahau": run_ekahau,
    "programmer": run_programmer,
}


async def chat(message, history):
    """Orchestrator LLM + specialist — full chat, not keyword pick_agent."""
    record_client_command(_text(message) or str(message))
    items = conversation_input(message, history)
    try:
        with trace("Orchestrator"):
            routed = await Runner.run(router_agent, items)
        choice = routed.final_output
        key = choice.agent if isinstance(choice, RouteChoice) else "wifi"
        reason = choice.reason if isinstance(choice, RouteChoice) else ""
        label = SPECIALISTS[key].name
        print(f"Router → {label} ({reason})")
        print(f"Conversation turns sent: {len(items)}")
        with trace(label):
            output = await RUNNERS[key](items)
        return output or "(The model returned no text. Open the notebook for the traceback.)"
    except Exception as e:
        print(f"chat() failed: {type(e).__name__}: {e}")
        return f"Error: {type(e).__name__}: {e}"



### Run the following and ask these prompts:

Note: click on the section that you want to copy and paste in the chat window.

1. Check which Ekahau .esx files are in Files/
2. Extract AP images from my Ekahau (file name) survey.



Then ask to check the AP model and vendor as this:


3. Can you list my Hamina OpenIntent files?
4. What are the AP models and vendors in my Hamina file: (OpenIntent File Name)?
5. Can you provide more detail on "AP NAME"?
6. Can you apply these changes to (Access Point Name) Radio settings:

2.4 GHz
Channel: 6
Transmit power: 9

5 GHz
Channel: 44
Transmit power: 9

6 GHz
Channel: 5
Transmit power: 9



Upload the file and check if it works.
Did it work? Changes were applied?
If yes, you're good! If no, was it surprising?

Important: Sometimes the agent is not the issue, the nature of the program/protocol/process requires more studies and insights.

If no, then you open the Hamina json file, you'll see the changes are applied as you asked:

```text

      "dot11_radios": [
        {
          "id": 0,
          "radio_function": "CLIENT_ACCESS",
          "band": "FREQ_2.4GHZ",
          "channel": 6,
          "channel_width": "20_MHz",
          "transmit_power": 9,
          "mimo_chains": 2,
          "channel_assignment": "AUTOMATIC"
        },
```

Sometimes the issue is on the **layer 8!**

Since the channel_assignment is set to AUTOMATIC; hence, Hamina takes over!
Changing the channel MANUAL mode is written/mentioned in the body of the prompt. Even though the programmer considers this, there might be a chance it doesn't interpret directly to the code, then you need to ask in the below format:



### Let's try again:




7. Apply these changes to (Access Point Name) Radio settings:

2.4 GHz
Channel: 6
Transmit power: 9
channel_assignment: "MANUAL"

5 GHz
Channel: 44
Transmit power: 9
channel_assignment: "MANUAL"

6 GHz
Channel: 5
Transmit power: 9
channel_assignment: "MANUAL"

In [ ]:
# Interrupt any old Gradio first. This launch uses chat() from the Orchestrator cell.
if "Orchestrator LLM" not in (getattr(chat, "__doc__", "") or ""):
    raise RuntimeError("Run the Orchestrator cell above before launching Gradio.")

# Do not use prevent_thread_lock=True with MCPServerStdio.
# Close the previous chat so the old orchestrator cannot keep answering on another port.
gr.close_all()
gr.ChatInterface(chat).launch(inbrowser=True)

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/aiagent2.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Lab 1 - PART 4 - A Second (External) MCP server</h2>
            <span style="color:#ff7800;">Add another MCP Server that already exists.<br/>
            In this section, we try to use an existing MCP server, install it in our machine and have it interact with our own agent.
            </span>
        </td>
    </tr>
</table>


### This is how your orchestrator agent will look like in the end; equipped with all the agents

<p align="center">
  <img src="../assets/routeragent4.png" width="600" />
</p>

## Agent 8: NetBeez

### Live in class

We want to use NetBeez MCP Server from the following link:

https://github.com/netbeez/nb-mcp-server

The reference document includes Quick Start (Recommended) section which brings an easy and smooth installation. The installation process will help users automatically set up the environment for Cursor, Claude Desktop, Codex, Windsurf and Kiro.<br>

However, this lab aims at installing an existing MCP server manually and have it talk to the Wi-Fi assistant we just built.<br>

In order to make it happen; we need to follow some steps together:<br>

### 1. Clone or download the repository
a. Open Terminal (in Mac) or CMD (in Windows), then navigate to the main folder "agnets-main"<br>

b. Copy and paste this: git clone https://github.com/netbeez/nb-mcp-server.git <br>

c. Copy and paste the Token and corresponding URL into your .env file (we skip this, I already placed them for you)<br>

### 2. Install dependencies & build

cd nb-mcp-server;

npm install;

npm run build;

npm run;


In [ ]:
# Start the NetBeez MCP as a child Node process, then list its tools.
# Node does not read .env files; it only sees the env dict we pass below.

# Folder that contains dist/index.js (sibling of 2_mcp/).
nb_dir = (Path("..") / "nb-mcp-server").resolve()

# Compiled MCP entrypoint (after npm run build).
nb_js = str(nb_dir / "dist" / "index.js")
if not Path(nb_js).is_file():
    raise FileNotFoundError(nb_js)

# Fail here if Node would otherwise exit immediately (McpError: Connection closed).
missing = [k for k in ("NETBEEZ_BASE_URL", "NETBEEZ_API_KEY") if not os.getenv(k)]
if missing:
    raise RuntimeError(f"Missing {missing}. Put them in the repo-root .env")

# Copy this kernel's env as str→str for subprocess so Node inherits NETBEEZ_* and PATH.
# You need it because Node is a separate process. It does not see Jupyter’s .env load,
# and it does not automatically get Python’s os.environ unless you pass env= into MCPServerStdio.

child_env = {k: str(v) for k, v in os.environ.items()}
# Force stdio (do not start the HTTP transport).
child_env["MCP_TRANSPORT"] = "stdio"

# Re-running this cell: close the previous Node MCP so we do not leak processes.
if "nb_mcp" in globals():
    try:
        await nb_mcp.cleanup()
    except Exception:
        pass

# Same pattern as Mist: Agents SDK speaks MCP over stdin/stdout of this process.
nb_mcp = MCPServerStdio(
    params={
        "command": "node",
        "args": [nb_js],
        "env": child_env,
        "cwd": str(nb_dir),
    },
    client_session_timeout_seconds=120,
    cache_tools_list=True,
)
await nb_mcp.connect()
for t in await nb_mcp.list_tools():
    print(t.name)



### Wasn't it interesting?
We didn't write any tools!<br>
It has already educated my agent about the list of actions. <br>
We don't have to provide any documentation. <br>
Publicly available. <br>


### Defining the NetBeez Agent:

In [70]:
NETBEEZ_INSTRUCTIONS = """
You use the connected NetBeez MCP tools for agents, tests, targets, alerts, Wi-Fi.
Call list_agents, list_tests, list_incidents, etc. Do not invent IDs.
"""

netbeez_agent = Agent(
    name="NetBeez Agent",
    instructions=NETBEEZ_INSTRUCTIONS,
    model=MODEL_NAME1,
    model_settings=ModelSettings(tool_choice="auto"),
    mcp_servers=[nb_mcp],
)

### Let's run it (our big orchestrator) for the last time while it's equipped with all the agents 🤩


In [ ]:

# Libraries are already imported at the top of the notebook


class RouteChoice(BaseModel):
    agent: Literal["wifi", "explorer", "programmer", "controller", "ekahau", "Hamina", "netbeez"]
    reason: str


SPECIALISTS = {
    "wifi": WiFi_assistant_agent,
    "explorer": explorer_agent,
    "programmer": programmer_assistant_agent,
    "controller": mcp_agent,
    "ekahau": ekahau_agent,
    "Hamina": Hamina_agent,
    "netbeez": netbeez_agent,
}

ORCHESTRATOR_INSTRUCTIONS = f"""
You are a router. You do not answer the user and you do not call APIs.
Choose exactly one specialist for this turn.

Match the conversation (all turns, not just the last line) to the specialist
whose instructions fit the work. Follow-ups stay with the same specialist
when they continue that job ("the list", "SSIDs", "do an API call", "from the MCP server",
"the zip", "OpenIntent", "AP model").
Listing files in Files/ (csv, pcap, pcapng, or "what files do I have") → wifi. That agent calls list_files.
.esx in Files/ → ekahau. OpenIntent .zip in Files/ → Hamina.
Explorer only fills a checklist from a named scan CSV in Explorer/.

wifi ({WiFi_assistant_agent.name}):
{WiFi_assistant_agent.instructions}

programmer ({programmer_assistant_agent.name}):
{programmer_assistant_agent.instructions}  

explorer ({explorer_agent.name}):
{explorer_agent.instructions}

controller ({mcp_agent.name}):
{mcp_agent.instructions}

Hamina ({Hamina_agent.name}):
{Hamina_agent.instructions}

ekahau ({ekahau_agent.name}):
{ekahau_agent.instructions}

netbeez ({netbeez_agent.name}):
{netbeez_agent.instructions}

"""

#Choosing a more powerful model to make the router more accurate

orchestrator_agent = Agent(
    name="Orchestrator Agent",
    instructions=ORCHESTRATOR_INSTRUCTIONS,
    model=MODEL_NAME3,
    output_type=RouteChoice,
)


def _text(content) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content.strip()
    if isinstance(content, dict):
        return _text(content.get("text") or content.get("content") or "")
    if isinstance(content, list):
        return "\n".join(part for part in (_text(x) for x in content) if part)
    return str(content).strip()


def conversation_input(message, history) -> list[dict]:
    items: list[dict] = []
    for item in history or []:
        if isinstance(item, dict):
            role = item.get("role") or "user"
            if role not in ("user", "assistant"):
                role = "user"
            text = _text(item.get("content") or item.get("text"))
            if text:
                items.append({"role": role, "content": text})
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            u, a = _text(item[0]), _text(item[1])
            if u:
                items.append({"role": "user", "content": u})
            if a:
                items.append({"role": "assistant", "content": a})
    text = _text(message) or str(message)
    items.append({"role": "user", "content": text})
    return items


async def run_wifi_assistant(items):
    result = await Runner.run(WiFi_assistant_agent, items)
    return result.final_output

async def run_programmer(items):
    result = await Runner.run(programmer_assistant_agent, items)
    return result.final_output

async def run_explorer(items):
    result = await Runner.run(explorer_agent, items)
    return result.final_output


async def run_controller(items):
    result = await Runner.run(mcp_agent, items)
    return result.final_output

async def run_Hamina(items):
    result = await Runner.run(Hamina_agent, items)
    return result.final_output

async def run_ekahau(items):
    result = await Runner.run(ekahau_agent, items)
    return result.final_output

async def run_netbeez(items):
    result = await Runner.run(netbeez_agent, items)
    return result.final_output

RUNNERS = {
    "wifi": run_wifi_assistant,
    "explorer": run_explorer,
    "programmer": run_programmer,
    "controller": run_controller,
    "ekahau": run_ekahau,
    "Hamina": run_Hamina,
    "netbeez": run_netbeez,
}


async def chat(message, history):
    """Orchestrator LLM + specialist — full chat, not keyword pick_agent."""
    record_client_command(_text(message) or str(message))
    items = conversation_input(message, history)
    try:
        with trace("Orchestrator"):
            routed = await Runner.run(router_agent, items)
        choice = routed.final_output
        key = choice.agent if isinstance(choice, RouteChoice) else "wifi"
        reason = choice.reason if isinstance(choice, RouteChoice) else ""
        label = SPECIALISTS[key].name
        print(f"Router → {label} ({reason})")
        print(f"Conversation turns sent: {len(items)}")
        with trace(label):
            output = await RUNNERS[key](items)
        return output or "(The model returned no text. Open the notebook for the traceback.)"
    except Exception as e:
        print(f"chat() failed: {type(e).__name__}: {e}")
        return f"Error: {type(e).__name__}: {e}"

# Interrupt any old Gradio first. This launch uses chat() from the Orchestrator cell.
if "Orchestrator LLM" not in (getattr(chat, "__doc__", "") or ""):
    raise RuntimeError("Run the Orchestrator cell above before launching Gradio.")

# Do not use prevent_thread_lock=True with MCPServerStdio.
# Close the previous chat so the old orchestrator cannot keep answering on another port.
gr.close_all()
gr.ChatInterface(chat).launch(inbrowser=True)


### Some sample tasks for NetBeez Agent

1. Can you check what frequency bands Amin_Home SSID is broadcasting using Netbeez tools?

2. Can you check and List all my NetBeez agents using Netbeez tools?

3. Can you check all my NetBeez agents network interfaces using Netbeez tools?